# LeetCode 1997 — First Day Where You Have Been in All the Rooms bằng Python

Link đề: https://leetcode.com/problems/first-day-where-you-have-been-in-all-the-rooms/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def firstDayBeenInAllRooms(self, nextVisit: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc hiểu một quy tắc di chuyển có **trạng thái ẩn** (tính chẵn/lẻ số lần đã vào mỗi phòng) và mô phỏng nó chính xác.
2. Phát hiện **bất biến then chốt**: lần đầu tiên bước vào phòng `i`, mọi phòng `0..i-1` đều đã được vào **chẵn** lần.
3. Xây dựng công thức quy hoạch động `f[i] = 2·f[i-1] − f[nextVisit[i-1]] + 2` (ngày đầu tiên vào phòng `i`) và một cách nhìn tương đương bằng **tổng tiền tố** của "độ dài vòng lặp".
4. Xử lý **modulo** `10⁹ + 7` khi có phép trừ, hiểu vì sao Python không bị số âm như Java/C++.
5. Kiểm chứng công thức bằng mô phỏng vét cạn trên dữ liệu ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Có `n` phòng đánh số `0..n-1`, các ngày đánh số từ `0`. Mỗi ngày bạn vào đúng **một** phòng.

- Ngày `0` bạn vào phòng `0`.
- Giả sử hôm nay bạn vào phòng `i` (tính cả lần này):
  - nếu tổng số lần đã vào phòng `i` là **lẻ** → hôm sau vào phòng `nextVisit[i]` (với `0 ≤ nextVisit[i] ≤ i`, tức là lùi lại hoặc đứng yên);
  - nếu tổng số lần là **chẵn** → hôm sau vào phòng `(i + 1) mod n`.

Trả về **số hiệu ngày đầu tiên** mà bạn **đã từng vào tất cả các phòng**. Đáp án có thể rất lớn nên trả về modulo `10⁹ + 7`.

**Ví dụ 1:** `nextVisit = [0,0]` → `2`. Ngày 0: phòng 0 (lần 1, lẻ → về 0). Ngày 1: phòng 0 (lần 2, chẵn → sang 1). Ngày 2: phòng 1 — đã vào đủ.

**Ví dụ 2:** `nextVisit = [0,0,2]` → `6`. Thứ tự phòng: `[0,0,1,0,0,1,2,...]`.

**Ví dụ 3:** `nextVisit = [0,1,2,0]` → `6`. Thứ tự phòng: `[0,0,1,1,2,2,3,...]`.

**Ràng buộc:** `n == len(nextVisit)`, `2 ≤ n ≤ 10⁵`, `0 ≤ nextVisit[i] ≤ i`.

In [1]:
import random
import time
from typing import List

MOD = 10**9 + 7
examples = [([0, 0], 2), ([0, 0, 2], 6), ([0, 1, 2, 0], 6)]
for nv, exp in examples:
    print(f"nextVisit={nv} -> kỳ vọng {exp}")

nextVisit=[0, 0] -> kỳ vọng 2
nextVisit=[0, 0, 2] -> kỳ vọng 6
nextVisit=[0, 1, 2, 0] -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Đếm số lần bằng mảng và kiểm tra chẵn/lẻ

`cnt[i]` lưu số lần đã vào phòng `i`. `x % 2 == 1` nghĩa là lẻ, `x % 2 == 0` là chẵn. Ta cũng có thể dùng `x & 1`.

In [2]:
cnt = [0] * 3
for room in [0, 0, 1, 0]:
    cnt[room] += 1
    print(f"vào phòng {room}: đã vào {cnt[room]} lần -> {'lẻ' if cnt[room] % 2 else 'chẵn'} (x & 1 = {cnt[room] & 1})")

vào phòng 0: đã vào 1 lần -> lẻ (x & 1 = 1)
vào phòng 0: đã vào 2 lần -> chẵn (x & 1 = 0)
vào phòng 1: đã vào 1 lần -> lẻ (x & 1 = 1)
vào phòng 0: đã vào 3 lần -> lẻ (x & 1 = 1)


### 3.2 Quy hoạch động (DP) một chiều

DP = lưu đáp án của các **bài con nhỏ hơn** trong mảng rồi dùng chúng để tính bài con lớn hơn. Ví dụ quen thuộc: `f[i] = f[i-1] + f[i-2]` (Fibonacci). Ở bài này `f[i]` sẽ phụ thuộc vào `f[i-1]` và một phần tử **bất kỳ phía trước** `f[nextVisit[i-1]]` — vẫn là DP một chiều vì chỉ nhìn về bên trái.

In [3]:
f = [0, 1]
for i in range(2, 10):
    f.append(f[i - 1] + f[i - 2])
print("Fibonacci:", f)

Fibonacci: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### 3.3 Tổng tiền tố (prefix sum)

`P[k] = a[0] + ... + a[k-1]` (với `P[0] = 0`). Khi đó tổng đoạn `a[l..r-1] = P[r] - P[l]` tính trong `O(1)`.

In [4]:
a = [3, 1, 4, 1, 5]
P = [0]
for x in a:
    P.append(P[-1] + x)
print("P =", P)
print("tổng a[1..3] =", sum(a[1:4]), "=", P[4] - P[1])

P = [0, 3, 4, 8, 9, 14]
tổng a[1..3] = 6 = 6


### 3.4 Modulo với phép trừ trong Python

Trong Python, `x % m` **luôn không âm** khi `m > 0`, kể cả khi `x < 0`. Ở Java/C++ thì `(-3) % 7 == -3`, nên người ta phải viết `(x % m + m) % m`. Ngoài ra, số nguyên Python **không tràn**, nên ta có thể tính số chính xác (rất lớn) để kiểm tra, rồi mới lấy modulo.

In [5]:
print((-3) % 7, "| (5 - 9) % 7 =", (5 - 9) % 7)
x = 2**200
print("2^200 có", len(str(x)), "chữ số; mod =", x % MOD)
print("lấy mod từng bước cũng ra cùng kết quả:", (pow(2, 200, MOD)) == x % MOD)

4 | (5 - 9) % 7 = 3
2^200 có 61 chữ số; mod = 499445072
lấy mod từng bước cũng ra cùng kết quả: True


## 4. Ý tưởng nền tảng / trực giác

Hãy **mô phỏng bằng tay** ví dụ 2: `nextVisit = [0,0,2]`.

| Ngày | Phòng | Số lần vào | Chẵn/lẻ | Hôm sau |
|---|---|---|---|---|
| 0 | 0 | 1 | lẻ | `nextVisit[0]=0` |
| 1 | 0 | 2 | chẵn | 1 |
| 2 | 1 | 1 | lẻ | `nextVisit[1]=0` |
| 3 | 0 | 3 | lẻ | 0 |
| 4 | 0 | 4 | chẵn | 1 |
| 5 | 1 | 2 | chẵn | 2 |
| 6 | 2 | 1 | — | **đủ phòng** |

Quan sát:
- Muốn **rời** phòng `i` sang `i+1`, số lần vào `i` phải **chẵn**.
- Lần đầu chạm phòng `i`, bạn **luôn bị đá lùi** về `nextVisit[i]` (vì lần 1 là lẻ). Rồi bạn phải "đi lại" từ `nextVisit[i]` lên tới `i`.
- Phòng chỉ lớn hơn khi đi kiểu `i → i+1`, nên ngày đầu vào phòng `n-1` chính là ngày đầu tiên đã vào đủ mọi phòng.

Câu hỏi: đoạn "đi lại" từ `nextVisit[i]` lên `i` tốn bao nhiêu ngày? Nếu nó **giống hệt** lần đi đầu tiên thì ta có thể dùng lại số ngày cũ!

In [6]:
def visit_order(nextVisit, days):
    n = len(nextVisit)
    cnt = [0] * n
    room, order = 0, []
    for _ in range(days):
        order.append(room)
        cnt[room] += 1
        room = nextVisit[room] if cnt[room] % 2 == 1 else (room + 1) % n
    return order

for nv, _ in examples:
    print(nv, "->", visit_order(nv, 8))

[0, 0] -> [0, 0, 1, 0, 0, 1, 0, 0]
[0, 0, 2] -> [0, 0, 1, 0, 0, 1, 2, 2]
[0, 1, 2, 0] -> [0, 0, 1, 1, 2, 2, 3, 0]


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng ngày

**Ý tưởng:** làm đúng như đề: giữ `cnt[]`, phòng hiện tại, số phòng khác nhau đã vào. Dừng khi số phòng đã vào bằng `n`.

In [7]:
def first_day_simulate(nextVisit):
    n = len(nextVisit)
    cnt = [0] * n
    seen = 0
    room, day = 0, 0
    while True:
        if cnt[room] == 0:
            seen += 1
        cnt[room] += 1
        if seen == n:
            return day
        room = nextVisit[room] if cnt[room] % 2 == 1 else (room + 1) % n
        day += 1

for nv, exp in examples:
    got = first_day_simulate(nv)
    print(nv, "->", got, "| kỳ vọng", exp)
    assert got == exp

[0, 0] -> 2 | kỳ vọng 2
[0, 0, 2] -> 6 | kỳ vọng 6
[0, 1, 2, 0] -> 6 | kỳ vọng 6


**Độ phức tạp:** Time = `O(D)` với `D` là đáp án (số ngày thật, chưa lấy mod), Space `O(n)`.

**Nút thắt:** `D` có thể tăng theo **hàm mũ**. Với `nextVisit = [0,0,...,0]`, mỗi lần chạm phòng mới bạn bị đá về phòng 0 và phải đi lại từ đầu, nên số ngày gần như **gấp đôi** mỗi khi thêm một phòng. Xem số liệu dưới đây.

In [8]:
for n in range(2, 17, 2):
    t = time.perf_counter()
    d = first_day_simulate([0] * n)
    print(f"n={n:2d}: đáp án = {d:7d}  (~2^{d.bit_length() - 1}),  mô phỏng mất {time.perf_counter() - t:.4f}s")

n= 2: đáp án =       2  (~2^1),  mô phỏng mất 0.0000s
n= 4: đáp án =      14  (~2^3),  mô phỏng mất 0.0000s
n= 6: đáp án =      62  (~2^5),  mô phỏng mất 0.0000s
n= 8: đáp án =     254  (~2^7),  mô phỏng mất 0.0000s
n=10: đáp án =    1022  (~2^9),  mô phỏng mất 0.0001s
n=12: đáp án =    4094  (~2^11),  mô phỏng mất 0.0003s
n=14: đáp án =   16382  (~2^13),  mô phỏng mất 0.0015s
n=16: đáp án =   65534  (~2^15),  mô phỏng mất 0.0058s


Với `n = 10⁵`, đáp án cỡ `2^100000` ngày — mô phỏng là **bất khả thi**. Ta cần một công thức tính số ngày mà không đi từng ngày.

## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến (invariant):** *Tại thời điểm lần đầu tiên bước vào phòng `i`, mọi phòng `0..i-1` đều đã được vào một số **chẵn** lần.*

Lý do: để đi từ phòng `j` sang phòng `j+1`, số lần vào `j` phải chẵn. Sau khi rời `j` lần cuối (ngay trước khi lên `j+1`), ta chỉ có thể quay lại `j` bằng cách bị đá lùi; mỗi lần bị đá lùi xuống ≤ `j` thì phải đi lại qua `j` và lại rời `j` với số lần chẵn. Vậy khi lần đầu chạm `i` (là phòng lớn nhất từng chạm), mọi phòng nhỏ hơn đang ở trạng thái "chẵn".

**Hệ quả quan trọng:** "chẵn" cũng giống như "chưa vào lần nào" về mặt hành vi: lần vào tiếp theo sẽ là lần **lẻ**, đúng như lần đầu tiên. Do đó, khi bị đá về `nextVisit[i]` rồi đi lên lại, hành trình từ `nextVisit[i]` tới `i` **lặp lại y hệt** hành trình lần đầu từ `nextVisit[i]` tới `i`.

Gọi `f[i]` = ngày đầu tiên vào phòng `i` (`f[0] = 0`). Quy tắc "nếu… thì…":
- **Nếu** đang ở `i-1` lần đầu (ngày `f[i-1]`) **thì** hôm sau (+1 ngày) bị đá về `j = nextVisit[i-1]`.
- Từ `j` lần "mới" đi lên `i-1` mất đúng `f[i-1] − f[j]` ngày (giống lần đầu).
- Tới `i-1` lần 2 (chẵn) **thì** +1 ngày sang `i`.

$$f[i] = f[i-1] + 1 + (f[i-1] - f[j]) + 1 = 2f[i-1] - f[\text{nextVisit}[i-1]] + 2.$$

Đáp án là `f[n-1] mod (10⁹+7)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP với số nguyên chính xác (không mod)

**Giải thích từng đoạn:**
1. `f = [0] * n` — `f[0] = 0` vì ngày 0 vào phòng 0.
2. Với `i = 1..n-1`: `j = nextVisit[i-1]`, `f[i] = 2*f[i-1] - f[j] + 2`.
3. Trả về `f[n-1]`.

Vì Python có số nguyên lớn, đây là bản **chính xác tuyệt đối**, rất tiện để đối chiếu với mô phỏng.

In [9]:
def first_day_exact(nextVisit):
    n = len(nextVisit)
    f = [0] * n
    for i in range(1, n):
        j = nextVisit[i - 1]
        f[i] = 2 * f[i - 1] - f[j] + 2
    return f[n - 1]

for nv, exp in examples:
    print(nv, "->", first_day_exact(nv)); assert first_day_exact(nv) == exp

[0, 0] -> 2
[0, 0, 2] -> 6
[0, 1, 2, 0] -> 6


Nhược điểm: số `f[i]` có thể dài tới ~`10⁵` bit, mỗi phép cộng/trừ tốn `O(độ dài)` ⇒ tổng có thể tới `O(n²/64)` thao tác máy. Vẫn chạy được với `n = 10⁵` nhưng chậm và tốn bộ nhớ. Cho LeetCode, ta lấy mod mỗi bước.

### 7.2 Cách 3 — DP lấy modulo từng bước (bản nộp chính)

Do mọi phép toán là cộng/trừ/nhân hằng, ta có thể lấy `% MOD` sau **mỗi** bước. Trong Python, `(2*a - b + 2) % MOD` luôn không âm nên không cần cộng thêm `MOD`.

In [10]:
def first_day_mod(nextVisit):
    n = len(nextVisit)
    f = [0] * n
    for i in range(1, n):
        f[i] = (2 * f[i - 1] - f[nextVisit[i - 1]] + 2) % MOD
    return f[-1]

for nv, exp in examples:
    assert first_day_mod(nv) == exp
print([first_day_mod(nv) for nv, _ in examples])

[2, 6, 6]


In [11]:
class SolutionDP:
    def firstDayBeenInAllRooms(self, nextVisit: List[int]) -> int:
        mod = 10**9 + 7
        n = len(nextVisit)
        f = [0] * n
        for i in range(1, n):
            f[i] = (2 * f[i - 1] - f[nextVisit[i - 1]] + 2) % mod
        return f[-1]

print([SolutionDP().firstDayBeenInAllRooms(nv) for nv, _ in examples])

[2, 6, 6]


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(n)` (mảng `f`; không thể chỉ giữ hai biến vì `f[nextVisit[i-1]]` có thể là phần tử bất kỳ phía trước).

### 7.3 Cách 4 — Nhìn bằng "độ dài vòng" và tổng tiền tố

Một cách nhìn khác, dễ tổng quát hơn: gọi `g[i]` = số ngày từ **lần đầu vào phòng `i`** tới **lần đầu vào phòng `i+1`**.

- Ngày đầu ở `i` → hôm sau ở `j = nextVisit[i]` (1 ngày).
- Từ `j` đi lên lại `i`: phải trải qua các đoạn `g[j], g[j+1], ..., g[i-1]` ⇒ `Σ_{k=j}^{i-1} g[k]` ngày.
- Ở `i` lần 2 → hôm sau lên `i+1` (1 ngày).

$$g[i] = 2 + \sum_{k=\text{nextVisit}[i]}^{i-1} g[k] = 2 + P[i] - P[\text{nextVisit}[i]],$$

với `P[k] = g[0] + ... + g[k-1]`. Ngày đầu vào phòng `n-1` là `P[n-1]`. Thật ra `P[i] = f[i]`, nên hai cách là một — nhưng góc nhìn "vòng lặp + tổng đoạn" giúp ta tự tin hơn khi gặp biến thể.

In [12]:
def first_day_prefix(nextVisit):
    n = len(nextVisit)
    P = [0] * n          # P[i] = ngày đầu vào phòng i = g[0] + ... + g[i-1]
    for i in range(n - 1):
        g = (2 + P[i] - P[nextVisit[i]]) % MOD
        P[i + 1] = (P[i] + g) % MOD
    return P[n - 1]

for nv, exp in examples:
    assert first_day_prefix(nv) == exp
print([first_day_prefix(nv) for nv, _ in examples])

[2, 6, 6]


In [13]:
class SolutionPrefix:
    def firstDayBeenInAllRooms(self, nextVisit: List[int]) -> int:
        mod = 10**9 + 7
        n = len(nextVisit)
        P = [0] * n
        for i in range(n - 1):
            g = 2 + P[i] - P[nextVisit[i]]
            P[i + 1] = (P[i] + g) % mod
        return P[n - 1]

print([SolutionPrefix().firstDayBeenInAllRooms(nv) for nv, _ in examples])

[2, 6, 6]


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace DP trên ví dụ 3 `nextVisit = [0,1,2,0]`:

| i | j = nextVisit[i-1] | f[i-1] | f[j] | vòng lặp = f[i-1] − f[j] | f[i] = 2f[i-1] − f[j] + 2 |
|---|---|---|---|---|---|
| 1 | 0 | 0 | 0 | 0 | 2 |
| 2 | 1 | 2 | 2 | 0 | 4 |
| 3 | 2 | 4 | 4 | 0 | 6 |

Mỗi phòng trỏ về chính nó, nên "vòng lặp" có độ dài 0: chỉ đứng lại một ngày rồi đi tiếp. Đáp án `6` ✓.

In [14]:
def trace_dp(nextVisit):
    n = len(nextVisit)
    f = [0] * n
    print(f"f[0] = 0")
    for i in range(1, n):
        j = nextVisit[i - 1]
        loop = f[i - 1] - f[j]
        f[i] = 2 * f[i - 1] - f[j] + 2
        print(f"i={i}: j=nextVisit[{i-1}]={j}, f[i-1]={f[i-1]}, f[j]={f[j]}, "
              f"vòng={loop} -> f[{i}] = {f[i-1]} + 1 + {loop} + 1 = {f[i]}")
    return f[-1]

print("Đáp án:", trace_dp([0, 1, 2, 0]))
print()
print("Đáp án:", trace_dp([0, 0, 2]))

f[0] = 0
i=1: j=nextVisit[0]=0, f[i-1]=0, f[j]=0, vòng=0 -> f[1] = 0 + 1 + 0 + 1 = 2
i=2: j=nextVisit[1]=1, f[i-1]=2, f[j]=2, vòng=0 -> f[2] = 2 + 1 + 0 + 1 = 4
i=3: j=nextVisit[2]=2, f[i-1]=4, f[j]=4, vòng=0 -> f[3] = 4 + 1 + 0 + 1 = 6
Đáp án: 6

f[0] = 0
i=1: j=nextVisit[0]=0, f[i-1]=0, f[j]=0, vòng=0 -> f[1] = 0 + 1 + 0 + 1 = 2
i=2: j=nextVisit[1]=0, f[i-1]=2, f[j]=0, vòng=2 -> f[2] = 2 + 1 + 2 + 1 = 6
Đáp án: 6


Đối chiếu với mô phỏng từng ngày: hàm `trace_simulate` in ra ngày, phòng, số lần và lý do chọn phòng hôm sau. Hãy để ý ngày **lần đầu** chạm mỗi phòng có khớp với `f[i]` ở trên không.

In [15]:
def trace_simulate(nextVisit):
    n = len(nextVisit)
    cnt = [0] * n
    first = [None] * n
    room, day = 0, 0
    while True:
        cnt[room] += 1
        if first[room] is None:
            first[room] = day
        if all(x is not None for x in first):
            print(f"ngày {day}: phòng {room} (lần {cnt[room]}) -> ĐÃ VÀO ĐỦ PHÒNG")
            break
        if cnt[room] % 2:
            nxt, why = nextVisit[room], "lẻ -> nextVisit"
        else:
            nxt, why = (room + 1) % n, "chẵn -> phòng kế"
        print(f"ngày {day}: phòng {room} (lần {cnt[room]}, {why}) -> mai vào {nxt}")
        room, day = nxt, day + 1
    print("ngày đầu vào từng phòng:", first)

trace_simulate([0, 0, 2])

ngày 0: phòng 0 (lần 1, lẻ -> nextVisit) -> mai vào 0
ngày 1: phòng 0 (lần 2, chẵn -> phòng kế) -> mai vào 1
ngày 2: phòng 1 (lần 1, lẻ -> nextVisit) -> mai vào 0
ngày 3: phòng 0 (lần 3, lẻ -> nextVisit) -> mai vào 0
ngày 4: phòng 0 (lần 4, chẵn -> phòng kế) -> mai vào 1
ngày 5: phòng 1 (lần 2, chẵn -> phòng kế) -> mai vào 2
ngày 6: phòng 2 (lần 1) -> ĐÃ VÀO ĐỦ PHÒNG
ngày đầu vào từng phòng: [0, 2, 6]


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def firstDayBeenInAllRooms(self, nextVisit: List[int]) -> int:
        MOD = 10**9 + 7
        n = len(nextVisit)
        f = [0] * n                      # f[i] = ngày đầu tiên vào phòng i
        for i in range(1, n):
            # đến i-1 lần đầu, bị đá về nextVisit[i-1], đi lại lên i-1, rồi sang i
            f[i] = (2 * f[i - 1] - f[nextVisit[i - 1]] + 2) % MOD
        return f[-1]

for nv, exp in examples:
    assert Solution().firstDayBeenInAllRooms(nv) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (thứ tự phòng mới).** Phòng có số hiệu lớn nhất đã từng vào chỉ tăng qua bước `i → i+1` (bước lùi `nextVisit[i] ≤ i` không tạo phòng mới lớn hơn; bước `(n-1) → 0` chỉ xảy ra sau khi đã vào phòng `n-1`). Vậy các phòng được khám phá theo thứ tự `0, 1, ..., n-1` và ngày cần tìm là `f[n-1]`.

**Bổ đề 2 (bất biến chẵn).** Ngay khi lần đầu vào phòng `i`, mọi phòng `k < i` có số lần vào chẵn. Chứng minh quy nạp theo thời gian: một phòng `k` chỉ được rời "lên trên" khi số lần chẵn; mọi lần quay lại `k` (do bị đá lùi) đều kết thúc bằng một lần rời `k` lên trên (vì để tới được `i` ta phải đi qua `k`), tức lại chẵn.

**Bổ đề 3 (lặp lại hành trình).** Hành vi của quá trình chỉ phụ thuộc vào phòng hiện tại và **tính chẵn lẻ** số lần vào các phòng. Khi bị đá về `j = nextVisit[i-1]` (từ lần đầu ở `i-1`), mọi phòng `j..i-2` đang chẵn — đúng trạng thái như lúc mới đi lần đầu (0 lần = chẵn). Phòng `i-1` đang lẻ (1 lần), nhưng hành trình từ `j` lên tới `i-1` không chạm `i-1` cho tới bước cuối. Do đó thời gian từ `j` đi lên tới `i-1` bằng đúng `f[i-1] − f[j]`.

Ghép lại: `f[i] = f[i-1] + 1 + (f[i-1] − f[j]) + 1`. Tại `i-1` lần 2 (chẵn) ta bước sang `i`. Quy nạp theo `i` cho thấy `f` đúng với mọi `i`. Lấy modulo không đổi kết quả cuối vì mọi phép là cộng/trừ/nhân với số nguyên.

Ta kiểm chứng **thực nghiệm** Bổ đề 2: mô phỏng nhiều mảng ngẫu nhiên, mỗi khi lần đầu chạm một phòng mới `i`, kiểm tra mọi `cnt[k]` với `k < i` đều chẵn. Đồng thời kiểm Bổ đề 1: thứ tự khám phá phòng mới là `0, 1, 2, ...`.

In [17]:
def check_invariant(nextVisit):
    n = len(nextVisit); cnt = [0] * n; order = []; room = 0
    while len(order) < n:
        if cnt[room] == 0:
            order.append(room)
            assert all(cnt[k] % 2 == 0 for k in range(room)), "bất biến chẵn bị vi phạm"
        cnt[room] += 1
        room = nextVisit[room] if cnt[room] % 2 else (room + 1) % n
    assert order == list(range(n)), "thứ tự khám phá không tăng dần"

rng = random.Random(10)
for _ in range(300):
    n = rng.randint(2, 11)
    check_invariant([rng.randint(0, i) for i in range(n)])
print("Bổ đề 1 và Bổ đề 2 đúng trên 300 mảng ngẫu nhiên")

Bổ đề 1 và Bổ đề 2 đúng trên 300 mảng ngẫu nhiên


## 11. Phân tích Time / Space

Gọi `n = len(nextVisit)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Mô phỏng | `O(D)`, `D` có thể ~`2^n` | `O(n)` | chỉ dùng kiểm tra, `n ≤ ~16` |
| DP chính xác (số lớn) | `O(n · n/w)` (w = kích thước từ máy) | `O(n²)` bit | đúng tuyệt đối, chậm |
| DP mod | `O(n)` | `O(n)` | bản nộp |
| Prefix "độ dài vòng" | `O(n)` | `O(n)` | cùng bản chất |

Không có average/worst khác nhau: vòng lặp luôn chạy đúng `n-1` lần. Đo thử với `n = 10⁵`:

In [18]:
rng = random.Random(1997)
big = [rng.randint(0, i) for i in range(10**5)]
t = time.perf_counter(); r1 = Solution().firstDayBeenInAllRooms(big); t1 = time.perf_counter() - t
print("n = 10^5, DP mod:", r1, f"({t1:.3f}s)")
mid = big[:20000]   # số lớn tốn bộ nhớ ~ n^2 bit, nên chỉ thử n = 2*10^4
t = time.perf_counter(); r2 = first_day_exact(mid); t2 = time.perf_counter() - t
print("n = 2*10^4, DP số lớn: đáp án có", r2.bit_length(), "bit", f"({t2:.3f}s)")
assert r2 % MOD == Solution().firstDayBeenInAllRooms(mid)

n = 10^5, DP mod: 22888641 (0.014s)
n = 2*10^4, DP số lớn: đáp án có 19986 bit (0.026s)


## 12. Bộ test

Gồm: ví dụ đề, biên (`n = 2`), các dạng đặc biệt (toàn 0, `nextVisit[i] = i`), và đối chiếu 4 cách với mô phỏng trên 400 mảng ngẫu nhiên (`n ≤ 12`, seed cố định, đúng ràng buộc `0 ≤ nextVisit[i] ≤ i`).

In [19]:
sols = {
    "exact": lambda a: first_day_exact(a) % MOD,
    "mod": first_day_mod,
    "prefix": first_day_prefix,
    "Solution": lambda a: Solution().firstDayBeenInAllRooms(a),
    "SolutionPrefix": lambda a: SolutionPrefix().firstDayBeenInAllRooms(a),
}
# ví dụ đề
for nv, exp in examples:
    for name, fn in sols.items():
        assert fn(nv) == exp, (name, nv)
# biên n = 2
assert Solution().firstDayBeenInAllRooms([0, 0]) == 2
assert Solution().firstDayBeenInAllRooms([0, 1]) == 2
# toàn 0: f[i] = 2 f[i-1] + 2  => f[i] = 2^(i+1) - 2
for n in range(2, 15):
    assert first_day_exact([0] * n) == 2 ** n - 2 == first_day_simulate([0] * n)
# nextVisit[i] = i: mỗi phòng 2 ngày
for n in range(2, 15):
    assert first_day_mod(list(range(n))) == 2 * (n - 1)
print("Ví dụ, biên, dạng đặc biệt: OK")

Ví dụ, biên, dạng đặc biệt: OK


In [20]:
rng = random.Random(42)
cnt = 0
for _ in range(400):
    n = rng.randint(2, 12)
    nv = [rng.randint(0, i) for i in range(n)]
    want = first_day_simulate(nv) % MOD
    for name, fn in sols.items():
        assert fn(nv) == want, (name, nv)
    cnt += 1
print(f"{cnt} test ngẫu nhiên: mọi cách khớp mô phỏng")

400 test ngẫu nhiên: mọi cách khớp mô phỏng


In [21]:
# test lớn: đáp án vượt MOD, so DP mod với DP số lớn
rng = random.Random(7)
for _ in range(5):
    n = rng.randint(2000, 5000)
    nv = [rng.randint(0, i) for i in range(n)]
    assert first_day_mod(nv) == first_day_exact(nv) % MOD == first_day_prefix(nv)
print("Test lớn (đáp án vượt mod): OK")

Test lớn (đáp án vượt mod): OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng nhầm chỉ số `nextVisit[i]` thay vì `nextVisit[i-1]`

Khi tính `f[i]`, phòng bị đá lùi là phòng `i-1` (nơi ta đang đứng lần đầu), nên phải dùng `nextVisit[i-1]`.

In [22]:
def wrong_index(nextVisit):
    n = len(nextVisit)
    f = [0] * n
    for i in range(1, n):
        f[i] = (2 * f[i - 1] - f[nextVisit[i]] + 2) % MOD   # SAI: nextVisit[i]
    return f[-1]

nv = [0, 1, 0]
print("sai:", wrong_index(nv), "| đúng:", first_day_simulate(nv))
# lưu ý: với [0,0,2] bản sai lại tình cờ ra đúng 6 -> ví dụ đề không đủ để phát hiện lỗi
print("[0,0,2] (tình cờ đúng):", wrong_index([0, 0, 2]))

sai: 6 | đúng: 4
[0,0,2] (tình cờ đúng): 6


In [23]:
def fixed_index(nextVisit):
    n = len(nextVisit)
    f = [0] * n
    for i in range(1, n):
        f[i] = (2 * f[i - 1] - f[nextVisit[i - 1]] + 2) % MOD   # ĐÚNG
    return f[-1]

print("sửa:", fixed_index([0, 1, 0]))

sửa: 4


### Lỗi 2 — Quên rằng lần đầu ở mỗi phòng luôn bị đá lùi (nghĩ `f[i] = f[i-1] + 1`)

Nhiều bạn nghĩ "vào phòng i-1 rồi hôm sau sang i". Sai: lần đầu là lần **lẻ**, nên luôn bị đá về `nextVisit[i-1]`, kể cả khi `nextVisit[i-1] = i-1` (khi đó mất thêm 1 ngày đứng yên).

In [24]:
def wrong_plus_one(nextVisit):
    return len(nextVisit) - 1     # SAI: coi như mỗi ngày tiến thêm một phòng

for nv, exp in examples:
    print(nv, "sai:", wrong_plus_one(nv), "| đúng:", exp)

[0, 0] sai: 1 | đúng: 2
[0, 0, 2] sai: 2 | đúng: 6
[0, 1, 2, 0] sai: 3 | đúng: 6


In [25]:
def fixed_plus_one(nextVisit):
    return Solution().firstDayBeenInAllRooms(nextVisit)

print([fixed_plus_one(nv) for nv, _ in examples])

[2, 6, 6]


### Lỗi 3 — Mô phỏng trực tiếp rồi lấy mod (TLE)

Mô phỏng cho đáp án đúng nhưng số ngày tăng theo hàm mũ. Ta chạy thử có **giới hạn số bước** để thấy nó không dừng kịp với `n` chỉ cỡ 40.

In [26]:
def simulate_limited(nextVisit, limit):
    n = len(nextVisit); cnt = [0] * n; seen = 0; room = 0
    for day in range(limit):
        if cnt[room] == 0:
            seen += 1
        cnt[room] += 1
        if seen == n:
            return day
        room = nextVisit[room] if cnt[room] % 2 else (room + 1) % n
    return None

res = simulate_limited([0] * 40, 10**6)
print("n=40, sau 10^6 bước:", "chưa xong!" if res is None else res)
print("số ngày thật:", first_day_exact([0] * 40))

n=40, sau 10^6 bước: chưa xong!
số ngày thật: 1099511627774


In [27]:
print("DP cho n=40 tức thì:", Solution().firstDayBeenInAllRooms([0] * 40))

DP cho n=40 tức thì: 511620081


### Lỗi 4 — Tưởng chỉ cần giữ 2 biến như Fibonacci

`f[nextVisit[i-1]]` có thể là **bất kỳ** phần tử cũ nào, nên phải giữ cả mảng. Bản "tối ưu bộ nhớ" sai dưới đây chỉ nhớ `f[i-2]`.

In [28]:
def wrong_two_vars(nextVisit):
    prev2, prev1 = 0, 0          # cố gắng chỉ nhớ f[i-2], f[i-1]
    for i in range(1, len(nextVisit)):
        j = nextVisit[i - 1]
        fj = prev1 if j == i - 1 else prev2   # SAI khi j < i-2
        prev2, prev1 = prev1, (2 * prev1 - fj + 2) % MOD
    return prev1

nv = [0, 1, 2, 0, 0]
print("sai:", wrong_two_vars(nv), "| đúng:", first_day_simulate(nv) % MOD)

sai: 10 | đúng: 14


In [29]:
print("sửa (giữ cả mảng):", first_day_mod([0, 1, 2, 0, 0]))

sửa (giữ cả mảng): 14


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Đề có "đáp án rất lớn, lấy mod"** + quy tắc mô phỏng ⇒ gần như chắc chắn **không mô phỏng**; tìm công thức truy hồi.
- **Con trỏ chỉ lùi** (`nextVisit[i] ≤ i`) ⇒ phòng mới được khám phá theo thứ tự tăng dần ⇒ DP theo chỉ số phòng.
- **Trạng thái chẵn/lẻ "reset"** về giống ban đầu ⇒ hành trình lặp lại ⇒ có thể **tái sử dụng** thời gian đã tính (`f[i-1] − f[j]`).
- Viết **mô phỏng nhỏ** trước để in ra `f[i]` thực tế và đoán công thức; sau đó chứng minh.
- Ở Python: không cần `+ MOD` khi trừ, nhưng khi dịch sang Java/C++ nhớ `(x % MOD + MOD) % MOD`.

## 15. Biến thể của bài

1. **Ngày đầu vào mọi phòng** (trả cả mảng `f`) — chính là mảng DP.
2. **Ngày vào phòng `i` lần thứ 2**: `s[i] = f[i] + 1 + (f[i] − f[nextVisit[i]])`.
3. **Quy tắc "vào lần thứ k mới được đi tiếp"** (thay chẵn/lẻ bằng mod `k`): mỗi phòng bị đá lùi `k-1` lần ⇒ `f[i] = f[i-1] + (k-1)·(1 + f[i-1] − f[j]) + 1`.
4. **Số lần đã vào phòng 0** tính tới ngày `f[n-1]` — đếm bằng DP tương tự (bài tập 5 dùng ý tưởng gần giống).

Dưới đây kiểm chứng biến thể 3 với mô phỏng (biến thể tự dựng — tự kiểm bằng brute force).

In [30]:
def simulate_k(nextVisit, k):
    n = len(nextVisit); cnt = [0] * n; seen = 0; room = 0; day = 0
    while True:
        if cnt[room] == 0: seen += 1
        cnt[room] += 1
        if seen == n: return day
        room = (room + 1) % n if cnt[room] % k == 0 else nextVisit[room]
        day += 1

def first_day_k(nextVisit, k):
    n = len(nextVisit); f = [0] * n
    for i in range(1, n):
        j = nextVisit[i - 1]
        f[i] = f[i - 1] + (k - 1) * (1 + f[i - 1] - f[j]) + 1
    return f[-1]

rng = random.Random(3)
for _ in range(200):
    n = rng.randint(2, 7); k = rng.randint(2, 4)
    nv = [rng.randint(0, i) for i in range(n)]
    assert first_day_k(nv, k) == simulate_k(nv, k)
print("Biến thể 'k lần' khớp mô phỏng; k=2 trùng bài gốc:", first_day_k([0, 0, 2], 2))

Biến thể 'k lần' khớp mô phỏng; k=2 trùng bài gốc: 6


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | đi từng ngày theo luật | `O(D)` ~ `O(2^n)` | `O(n)` | kiểm tra, đoán công thức |
| 2. DP số lớn | `f[i] = 2f[i-1] − f[j] + 2` không mod | `O(n²/w)` | lớn | đối chiếu chính xác |
| 3. DP mod | như trên, mod từng bước | `O(n)` | `O(n)` | **nộp bài** |
| 4. Prefix độ dài vòng | `g[i] = 2 + P[i] − P[j]` | `O(n)` | `O(n)` | khi muốn tổng quát hóa |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `nextVisit = [0, 1, 2, 3, 4]` (mỗi phòng trỏ về chính nó). Đáp án là bao nhiêu?
2. **Tính tay:** điền bảng `f` cho `nextVisit = [0, 0, 0, 0]`.
3. **Điền TODO:** hoàn thiện `first_days_all` trả về **cả mảng** `f` (không mod).
4. **Tự viết lại:** viết lại lời giải bằng cách prefix "độ dài vòng" mà không nhìn code.
5. **Mini challenge:** viết `second_visit_day(nextVisit, i)` trả về ngày **lần thứ hai** vào phòng `i` (với `i ≤ n-2`, không mod). Kiểm bằng mô phỏng.

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def first_days_all(nextVisit):
    n = len(nextVisit)
    f = [0] * n
    # TODO: điền vòng for tính f[i]
    return f

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_prefix_solution(nextVisit):
    # TODO: P[i+1] = P[i] + (2 + P[i] - P[nextVisit[i]])
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def second_visit_day(nextVisit, i):
    # TODO: dùng f[i] và f[nextVisit[i]]
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Lần đầu vào phòng `i` bị đá về `i`. (2) Mỗi phòng tốn đúng 2 ngày. (3) `f[i] = 2i`.

**Bài 2.** (1) `f[0] = 0`. (2) Với `nextVisit` toàn 0, `f[j] = f[0] = 0`. (3) `f[i] = 2f[i-1] + 2` ⇒ `0, 2, 6, 14`.

**Bài 3.** (1) Bắt đầu từ `i = 1`. (2) `j = nextVisit[i-1]`. (3) `f[i] = 2*f[i-1] - f[j] + 2`.

**Bài 4.** (1) `P` có độ dài `n`, `P[0] = 0`. (2) Vòng `i` từ `0` tới `n-2`. (3) Trả `P[n-1]`.

**Bài 5.** (1) Lần đầu ở `i` là ngày `f[i]`, hôm sau ở `nextVisit[i]`. (2) Từ `nextVisit[i]` đi lên lại `i` mất `f[i] − f[nextVisit[i]]`. (3) `f[i] + 1 + f[i] − f[nextVisit[i]]`.

## 19. Đáp án tham khảo

In [34]:
# Bài 1
assert Solution().firstDayBeenInAllRooms([0, 1, 2, 3, 4]) == 8 == first_day_simulate([0, 1, 2, 3, 4])
# Bài 2
def first_days_all_ans(nextVisit):
    n = len(nextVisit); f = [0] * n
    for i in range(1, n):
        f[i] = 2 * f[i - 1] - f[nextVisit[i - 1]] + 2
    return f
assert first_days_all_ans([0, 0, 0, 0]) == [0, 2, 6, 14]
print("Bài 1: 8 | Bài 2:", first_days_all_ans([0, 0, 0, 0]))

Bài 1: 8 | Bài 2: [0, 2, 6, 14]


In [35]:
# Bài 3: kiểm mảng f với ngày đầu vào từng phòng trong mô phỏng
def first_visit_days_sim(nextVisit):
    n = len(nextVisit); cnt = [0] * n; first = [None] * n; room = 0; day = 0
    while None in first:
        cnt[room] += 1
        if first[room] is None: first[room] = day
        room = nextVisit[room] if cnt[room] % 2 else (room + 1) % n
        day += 1
    return first

# Bài 4
def my_prefix_solution_ans(nextVisit):
    n = len(nextVisit); P = [0] * n
    for i in range(n - 1):
        P[i + 1] = (P[i] + 2 + P[i] - P[nextVisit[i]]) % MOD
    return P[n - 1]

rng = random.Random(17)
for _ in range(200):
    n = rng.randint(2, 10); nv = [rng.randint(0, i) for i in range(n)]
    assert first_days_all_ans(nv) == first_visit_days_sim(nv)
    assert my_prefix_solution_ans(nv) == first_day_simulate(nv) % MOD
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [36]:
# Bài 5
def second_visit_day_ans(nextVisit, i):
    f = first_days_all_ans(nextVisit)
    return f[i] + 1 + f[i] - f[nextVisit[i]]

def second_visit_sim(nextVisit, i):
    n = len(nextVisit); cnt = [0] * n; room = 0; day = 0
    while True:
        cnt[room] += 1
        if room == i and cnt[room] == 2: return day
        room = nextVisit[room] if cnt[room] % 2 else (room + 1) % n
        day += 1

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(2, 9); nv = [rng.randint(0, i) for i in range(n)]
    i = rng.randint(0, n - 2)
    assert second_visit_day_ans(nv, i) == second_visit_sim(nv, i)
print("Bài 5: OK, ví dụ [0,0,2], i=1 ->", second_visit_day_ans([0, 0, 2], 1))

Bài 5: OK, ví dụ [0,0,2], i=1 -> 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
f = [0] * n                    # f[i] = ngày đầu vào phòng i
for i in range(1, n):
    j = nextVisit[i - 1]       # lần đầu ở i-1 bị đá về j
    f[i] = (2 * f[i - 1] - f[j] + 2) % MOD
return f[-1]
```

Nhớ bằng câu: **"tới i-1, bị đá về j, đi lại đúng quãng cũ, rồi bước sang i"** ⇒ `f[i-1] + 1 + (f[i-1] − f[j]) + 1`.

In [37]:
def template(nextVisit):
    f = [0] * len(nextVisit)
    for i in range(1, len(nextVisit)):
        f[i] = (2 * f[i - 1] - f[nextVisit[i - 1]] + 2) % MOD
    return f[-1]
print([template(nv) for nv, _ in examples])

[2, 6, 6]


**Checklist:**

- [ ] Dùng `nextVisit[i-1]` (không phải `nextVisit[i]`) khi tính `f[i]`.
- [ ] `f[0] = 0`, vòng lặp bắt đầu từ `i = 1`.
- [ ] Lấy `% MOD` mỗi bước; nếu dịch sang ngôn ngữ khác, cộng `MOD` trước khi `%` để tránh số âm.
- [ ] Giữ **cả mảng** `f` (không tối ưu xuống 2 biến).
- [ ] Trả về `f[n-1]`, không cộng thêm 1.
- [ ] Đã thử `n = 2`, mảng toàn 0 (đáp án `2^n − 2`), mảng `nextVisit[i] = i` (đáp án `2(n−1)`).